<a href="https://colab.research.google.com/github/Rodriguez-9819/Aaron_INFO4670_Fall2026/blob/main/Aaron_Rodriguez_Assignment2.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# INFO 4670 / 4760 — Assignment 2
### Cleaning & Integrating the Northgate Data — Completed

**Student:** Aaron Rodriguez

This notebook cleans and integrates `student_records.csv`, `course_enrollments.csv`, and `weekly_activity.csv` into one analysis-ready, one-row-per-student table. All cleaning is performed on copies; the raw files are never overwritten.

- Run the notebook top to bottom in Google Colab.
- The final code writes `northgate_clean.csv`.
- Submit the completed `.ipynb` through GitHub as required by the assignment.


## Setup — load the three files (given)

In [26]:
import pandas as pd, numpy as np, os

try:
    students_raw = pd.read_csv("student_records.csv")
except FileNotFoundError:
    from google.colab import files
    print("Upload student_records.csv, course_enrollments.csv, weekly_activity.csv")
    files.upload()
    students_raw = pd.read_csv("student_records.csv")

enroll_raw   = pd.read_csv("course_enrollments.csv")
activity_raw = pd.read_csv("weekly_activity.csv")

# Golden rule: work on copies, never overwrite the raw files.
students = students_raw.copy()
enroll   = enroll_raw.copy()
activity = activity_raw.copy()

print("students", students.shape, "| enroll", enroll.shape, "| activity", activity.shape)


students (2027, 12) | enroll (8088, 4) | activity (32000, 4)


## Part A · Clean student_records

### A1 · Missing values
Find how many values are missing in `study_hours_reported` and store the count as **`n_missing_study`**. Then, in the markdown cell after your code, say in 1–2 sentences which of Han's methods you would use to handle it and why.
*Hint:* `.isna().sum()`

In [6]:
# Count missing study-hours values before cleaning.
n_before = len(students)
n_missing_study = int(students["study_hours_reported"].isna().sum())
n_present_before = int(students["study_hours_reported"].notna().sum())

# The observed distribution is almost symmetric, so mean imputation is reasonable.
study_mean = students["study_hours_reported"].mean()
study_median = students["study_hours_reported"].median()
study_skew = students["study_hours_reported"].skew()

students["study_hours_reported"] = students["study_hours_reported"].fillna(study_mean)

n_after = len(students)
n_missing_after = int(students["study_hours_reported"].isna().sum())

print(f"Rows before: {n_before}; rows after: {n_after}")
print(f"Study hours present before: {n_present_before}; missing before: {n_missing_study}; missing after: {n_missing_after}")
print(f"Observed mean = {study_mean:.2f}, median = {study_median:.2f}, skew = {study_skew:.3f}")


Rows before: 2027; rows after: 2027
Study hours present before: 1772; missing before: 255; missing after: 0
Observed mean = 10.49, median = 10.50, skew = -0.066


**Your justification (1–2 sentences):** I used Han's **mean-value replacement** method for `study_hours_reported`. The observed values are approximately symmetric (mean ≈ 10.49, median = 10.50, skew ≈ -0.07), so the mean is a reasonable representative value; this keeps all 2,027 rows instead of dropping the 255 students with blanks.

In [7]:
# ✅ Check
try:
    assert n_missing_study == 255
    print("✅ A1 correct — 255 missing (n = 1772 present)")
except Exception:
    print("❌ A1 not yet — set n_missing_study to the count of blank study_hours_reported")

✅ A1 correct — 255 missing (n = 1772 present)


### A2 · Inconsistent categories
Standardize the `housing` column into its three real groups and store the result as a new column **`students["housing_clean"]`**.
*Hint:* `.str.strip().str.lower().map({...})`

In [8]:
# Show the inconsistent labels before cleaning.
print("Housing counts BEFORE:")
print(students["housing"].value_counts(dropna=False))

housing_key = students["housing"].astype(str).str.strip().str.lower()
housing_map = {
    "on-campus": "On-Campus",
    "off-campus": "Off-Campus",
    "off campus": "Off-Campus",
    "with family": "With Family"
}
students["housing_clean"] = housing_key.map(housing_map)

print("\nHousing counts AFTER:")
print(students["housing_clean"].value_counts(dropna=False))


Housing counts BEFORE:
housing
Off-Campus     755
On-Campus      480
With Family    420
off campus     113
Off-campus      77
with family     72
on-campus       69
 On-Campus      41
Name: count, dtype: int64

Housing counts AFTER:
housing_clean
Off-Campus     945
On-Campus      590
With Family    492
Name: count, dtype: int64


In [9]:
# ✅ Check
try:
    assert students["housing_clean"].nunique() == 3
    print("✅ A2 correct — 3 groups:", {k:int(v) for k,v in students["housing_clean"].value_counts().items()})
except Exception:
    print("❌ A2 not yet — housing_clean should have exactly 3 groups (expect 590 / 945 / 492)")

✅ A2 correct — 3 groups: {'Off-Campus': 945, 'On-Campus': 590, 'With Family': 492}


### A3 · Errors vs. extremes
Find the impossible values. Store the sorted unique impossible ages as **`impossible_ages`** and the number of rows with negative work hours as **`n_neg_work`**. (Remember: extreme-but-valid values like a long commute are *kept*.)
*Hint:* boolean masks on `age` and `work_hours_per_week`.

**Decision:** Ages below 16 or above 100 and negative work hours violate logical validity rules, so I treat them as data-entry errors. Because the source does not tell us the true values, I replace only those impossible values with the median of the valid values rather than deleting the students; an *impossible* value violates a rule (for example, -10 work hours), while an *extreme but valid* value is unusual but still possible and should be retained.

In [10]:
# Identify impossible values BEFORE fixing them.
age_invalid = ~students["age"].between(16, 100)
work_invalid = students["work_hours_per_week"] < 0

impossible_ages = sorted(students.loc[age_invalid, "age"].unique().tolist())
n_neg_work = int(work_invalid.sum())

# Fix the invalid cells without deleting student records.
age_median = int(students.loc[~age_invalid, "age"].median())
work_median = int(students.loc[~work_invalid, "work_hours_per_week"].median())

students.loc[age_invalid, "age"] = age_median
students.loc[work_invalid, "work_hours_per_week"] = work_median

print("Impossible ages found:", impossible_ages)
print("Negative work-hour rows:", n_neg_work)
print(f"Replacement values -> age median: {age_median}, work-hours median: {work_median}")
print("Impossible ages remaining:", int((~students["age"].between(16, 100)).sum()))
print("Negative work-hour rows remaining:", int((students["work_hours_per_week"] < 0).sum()))


Impossible ages found: [-22, 0, 1, 3, 199, 220]
Negative work-hour rows: 4
Replacement values -> age median: 25, work-hours median: 18
Impossible ages remaining: 0
Negative work-hour rows remaining: 0


In [11]:
# ✅ Check
try:
    assert 220 in impossible_ages and -22 in impossible_ages and n_neg_work == 4
    print("✅ A3 correct — impossible ages incl. -22/199/220; 4 negative work-hour rows")
except Exception:
    print("❌ A3 not yet — check ages (e.g. -22, 199, 220) and count negative work hours (expect 4)")

✅ A3 correct — impossible ages incl. -22/199/220; 4 negative work-hour rows


### A4 · Duplicates
Remove duplicate **student** records and store the result as **`students_dedup`**. Then, in the markdown cell after your code, explain in one sentence why you must NOT de-duplicate `enroll` or `activity` by ID.
*Hint:* `.drop_duplicates()` — think about exact vs. near-duplicates.

In [12]:
# Full-row duplicate detection catches the exact duplicates,
# but student_id catches both exact and near-duplicate student rows.
n_exact_duplicates = int(students.duplicated().sum())
n_duplicate_id_extras = int(students.duplicated(subset="student_id").sum())

students_dedup = students.drop_duplicates(subset="student_id", keep="first").copy()

print("Exact duplicate rows:", n_exact_duplicates)
print("Extra rows with a repeated student_id:", n_duplicate_id_extras)
print("Rows before de-duplication:", len(students))
print("Rows after de-duplication:", len(students_dedup))
print("Unique student IDs:", students_dedup["student_id"].nunique())


Exact duplicate rows: 18
Extra rows with a repeated student_id: 27
Rows before de-duplication: 2027
Rows after de-duplication: 2000
Unique student IDs: 2000


**Why not de-dupe enroll / activity? (1 sentence):** I do not de-duplicate those tables by student ID because each enrollment row represents a separate course taken and each activity row represents a separate student-week observation, so repeated IDs are expected legitimate records rather than duplicate students.

In [13]:
# ✅ Check
try:
    assert len(students_dedup) == 2000 and students_dedup["student_id"].is_unique
    print("✅ A4 correct — 2000 unique students (from 2027 rows)")
except Exception:
    print("❌ A4 not yet — students_dedup should be 2000 rows, one per student")

✅ A4 correct — 2000 unique students (from 2027 rows)


## Part B · Integrate the three files

### B5 · Standardize the key & integrate
Build one **row-per-student** analysis table called **`analysis`**: start from `students_dedup`, add a standardized numeric key, and merge in a per-student summary of `activity` (e.g., total `minutes_active`).
*Hint:* make the key with `.str.replace("NU-","")` → `int`; summarize activity with `groupby(...).sum()`; then `merge`.

In [14]:
# Standardize the join key in all three tables.
students_dedup["sid_clean"] = (
    students_dedup["student_id"].str.replace("NU-", "", regex=False).astype(int)
)
enroll["sid_clean"] = pd.to_numeric(enroll["sid"], errors="coerce").astype("Int64")
activity["sid_clean"] = (
    activity["student_id"].str.replace("NU-", "", regex=False).astype(int)
)

# course_enrollments is one row per course, so summarize it to one row per student.
course_summary = (
    enroll.groupby("sid_clean")
          .agg(
              course_enrollments=("course", "size"),
              unique_courses=("course", "nunique"),
              terms_enrolled=("term", "nunique")
          )
          .reset_index()
)

# weekly_activity is one row per student-week, so summarize it to one row per student.
activity_summary = (
    activity.groupby("sid_clean")
            .agg(
                total_minutes_active=("minutes_active", "sum"),
                total_lms_clicks=("lms_clicks", "sum"),
                active_weeks=("week", "nunique")
            )
            .reset_index()
)

# Left joins preserve the cleaned roster as the analysis population.
analysis = (
    students_dedup
    .merge(course_summary, on="sid_clean", how="left")
    .merge(activity_summary, on="sid_clean", how="left")
)

print("Clean roster rows:", len(students_dedup))
print("Course summary rows:", len(course_summary))
print("Activity summary rows:", len(activity_summary))
print("Analysis rows:", len(analysis))
print("Analysis has unique student IDs:", analysis["student_id"].is_unique)


Clean roster rows: 2000
Course summary rows: 2014
Activity summary rows: 2000
Analysis rows: 2000
Analysis has unique student IDs: True


In [15]:
# ✅ Check
try:
    assert len(analysis) == 2000 and analysis["student_id"].is_unique
    print("✅ B5 correct — one row per student, 2000 rows")
except Exception:
    print("❌ B5 not yet — analysis should have one row per student (2000)")

✅ B5 correct — one row per student, 2000 rows


### B6 · Verify the join
Report how many `enroll` rows match a student in your standardized key. Store the count as **`matched`**.
*Hint:* `enroll["sid"].isin(set_of_keys).sum()`

In [16]:
student_keys = set(students_dedup["sid_clean"])

enroll_match_mask = enroll["sid_clean"].isin(student_keys)
matched = int(enroll_match_mask.sum())
unmatched_rows = int((~enroll_match_mask).sum())
orphan_ids = int(enroll.loc[~enroll_match_mask, "sid_clean"].nunique())

activity_orphan_ids = int(
    activity.loc[~activity["sid_clean"].isin(student_keys), "sid_clean"].nunique()
)

print(f"Enrollment rows matched: {matched} of {len(enroll)}")
print(f"Unmatched enrollment rows: {unmatched_rows}")
print(f"Distinct orphan enrollment IDs: {orphan_ids}")
print(f"Distinct orphan activity IDs: {activity_orphan_ids}")
print(f"Final analysis rows: {len(analysis)}; cleaned roster rows: {len(students_dedup)}")
print("Final row count equals cleaned roster:", len(analysis) == len(students_dedup))


Enrollment rows matched: 8041 of 8088
Unmatched enrollment rows: 47
Distinct orphan enrollment IDs: 14
Distinct orphan activity IDs: 0
Final analysis rows: 2000; cleaned roster rows: 2000
Final row count equals cleaned roster: True


In [17]:
# ✅ Check
try:
    assert matched == 8041
    print("✅ B6 correct — 8041 of 8088 enrollment rows match (14 orphan IDs)")
except Exception:
    print("❌ B6 not yet — count enrollment rows whose sid is in your student keys (expect 8041)")

✅ B6 correct — 8041 of 8088 enrollment rows match (14 orphan IDs)


## Part C · Transform

### C7 · Parse the dates
Parse `enrollment_date` so no valid date is lost. Store the parsed series as **`dates_parsed`** and check the number of NaT (blanks).
*Hint:* `pd.to_datetime(..., format="mixed", errors="coerce")` — compare NaT before and after.

In [18]:
date_blanks_before = int(analysis["enrollment_date"].isna().sum())

# format="mixed" lets pandas recognize all three date styles in this dataset.
dates_parsed = pd.to_datetime(
    analysis["enrollment_date"],
    format="mixed",
    errors="coerce"
)

date_blanks_after = int(dates_parsed.isna().sum())
naive_lost = int(pd.to_datetime(analysis["enrollment_date"], errors="coerce").isna().sum())

analysis["enrollment_date"] = dates_parsed

print("Blank/NaT dates before parsing:", date_blanks_before)
print("Blank/NaT dates after mixed-format parsing:", date_blanks_after)
print("Dates a naive single-format parse would lose:", naive_lost)


Blank/NaT dates before parsing: 0
Blank/NaT dates after mixed-format parsing: 0
Dates a naive single-format parse would lose: 1449


In [19]:
# ✅ Check
try:
    assert dates_parsed.isna().sum() == 0
    print("✅ C7 correct — all dates parsed, 0 lost (a naive parse would lose ~1470)")
except Exception:
    print("❌ C7 not yet — parse every format so no valid date becomes NaT")

✅ C7 correct — all dates parsed, 0 lost (a naive parse would lose ~1470)


### C8 · Normalize & discretize
Add two columns to `analysis`: a **z-scored** numeric column stored as **`analysis["study_z"]`**, and a **GPA band** column stored as **`analysis["gpa_band"]`** (bin `final_gpa` into 4 bands).
*Hint:* z-score = `(x - x.mean()) / x.std()`; bands = `pd.cut(..., bins=[-0.01,1,2,3,4])`.

In [20]:
# Z-score the cleaned/imputed study-hours column.
analysis["study_z"] = (
    (analysis["study_hours_reported"] - analysis["study_hours_reported"].mean())
    / analysis["study_hours_reported"].std()
)

# Discretize final GPA into four 1-point bands.
analysis["gpa_band"] = pd.cut(
    analysis["final_gpa"],
    bins=[-0.01, 1, 2, 3, 4],
    labels=["0–1", "1–2", "2–3", "3–4"],
    include_lowest=True
)

print("study_z mean:", round(float(analysis["study_z"].mean()), 6))
print("\nGPA band counts:")
print(analysis["gpa_band"].value_counts().sort_index())


study_z mean: -0.0

GPA band counts:
gpa_band
0–1      70
1–2     528
2–3    1102
3–4     300
Name: count, dtype: int64


In [21]:
# ✅ Check
try:
    assert analysis["gpa_band"].nunique() == 4 and abs(analysis["study_z"].mean()) < 0.01
    print("✅ C8 correct — z-score (mean ≈ 0) and 4 GPA bands added")
except Exception:
    print("❌ C8 not yet — add a z-scored column and a 4-band gpa_band column")

✅ C8 correct — z-score (mean ≈ 0) and 4 GPA bands added


## Part D · Deliver & reflect

### D9 · Write the clean file
Write your clean `analysis` table to **`northgate_clean.csv`** (do NOT overwrite the raw files).
*Hint:* `.to_csv("northgate_clean.csv", index=False)`

In [22]:
analysis.to_csv("northgate_clean.csv", index=False)
print("Saved northgate_clean.csv")
print("Output shape:", analysis.shape)


Saved northgate_clean.csv
Output shape: (2000, 22)


In [23]:
# ✅ Check
try:
    assert os.path.exists("northgate_clean.csv")
    print("✅ D9 correct — northgate_clean.csv written (raw files untouched)")
except Exception:
    print("❌ D9 not yet — write analysis to northgate_clean.csv")

✅ D9 correct — northgate_clean.csv written (raw files untouched)


### D10 · Cleaning log
In the markdown cell below, list each decision you made above and a one-line justification for it (missing values, housing, impossible values, duplicates, key, dates). *This is graded — no code needed.*

**Your cleaning log:**
- **Missing study hours → mean imputation.** There were 255 blanks; the observed distribution was approximately symmetric (mean 10.49, median 10.50), so mean replacement preserved the full roster without strongly shifting the center.
- **Housing → standardized text to three groups.** I stripped spaces, normalized case, and mapped spelling variants to `On-Campus`, `Off-Campus`, and `With Family`, reducing eight raw spellings to three real categories.
- **Impossible age/work values → median replacement, not row deletion.** Ages outside a plausible student range and four negative work-hour values are invalid; because their true values are unknown, I replaced only those cells with the valid-column medians and kept extreme-but-possible values.
- **Student duplicates → one row per `student_id`.** De-duplicating on the ID removed 27 extra student rows (18 exact duplicates plus 9 near-duplicates), leaving 2,000 unique students; I did not de-duplicate enrollment or activity rows because repeated IDs represent different courses or weeks.
- **Join key → standardized numeric `sid_clean`.** I removed the `NU-` prefix from student/activity IDs and converted all keys to numeric values before joining.
- **Course and activity integration → aggregate first, then left-join.** I summarized course enrollments (course count/unique courses/terms) and weekly activity (total minutes/clicks/weeks) per student so the final table remained one row per student.
- **Join verification → kept the roster as the analysis population.** 8,041 of 8,088 enrollment rows matched; the 47 unmatched rows came from 14 orphan IDs, while the final table still contained exactly the 2,000 cleaned students.
- **Enrollment dates → mixed-format parsing.** I used `pd.to_datetime(..., format="mixed")`, which parsed every valid date with 0 dates lost.
- **GPA zeros → retained.** The data alone does not establish that 0.00 GPAs are errors, so I did not delete them.


### D11 · Payoff
Using your clean `analysis` table, report the **mean GPA** and **one relationship** you find interesting, then note in one sentence how cleaning changed the picture versus the raw data.

In [24]:
mean_gpa = analysis["final_gpa"].mean()
raw_mean_gpa = students_raw["final_gpa"].mean()
activity_gpa_corr = analysis["total_minutes_active"].corr(analysis["final_gpa"])

print(f"Mean GPA (clean data): {mean_gpa:.3f}")
print(f"Mean GPA (raw student rows): {raw_mean_gpa:.3f}")
print(f"Correlation: total minutes active vs. final GPA = {activity_gpa_corr:.3f}")

print(
    "Relationship: Students with more total LMS minutes tended to have higher final GPAs; "
    f"the Pearson correlation is r = {activity_gpa_corr:.3f}."
)
print(
    "Cleaning changed the picture slightly: removing duplicate student rows changed the mean GPA "
    f"from {raw_mean_gpa:.3f} in the raw file to {mean_gpa:.3f} in the one-row-per-student table, "
    "while also making joins and category summaries valid."
)


Mean GPA (clean data): 2.305
Mean GPA (raw student rows): 2.307
Correlation: total minutes active vs. final GPA = 0.796
Relationship: Students with more total LMS minutes tended to have higher final GPAs; the Pearson correlation is r = 0.796.
Cleaning changed the picture slightly: removing duplicate student rows changed the mean GPA from 2.307 in the raw file to 2.305 in the one-row-per-student table, while also making joins and category summaries valid.


In [25]:
# ✅ Check
print("(D11 is interpreted by your instructor — make sure your numbers and one-sentence takeaway are shown above.)")

(D11 is interpreted by your instructor — make sure your numbers and one-sentence takeaway are shown above.)
